In [1]:
import os
import io
import copy
import random
from tqdm.auto import tqdm 

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

from torchvision import datasets, transforms, models
from torchvision.models import resnet18, ResNet18_Weights

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [2]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.247, 0.243, 0.261])
])

train_dataset_full = datasets.CIFAR10(root='./cifar-10-batches-py', train=True,
                                      download=True, transform=transform)
test_dataset_full = datasets.CIFAR10(root='./cifar-10-batches-py', train=False,
                                     download=True, transform=transform)

Files already downloaded and verified
Files already downloaded and verified


In [3]:
train_dataset = train_dataset_full
test_dataset = test_dataset_full

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("train - test dataset ready.")

train - test dataset ready.


In [4]:
class Encoder(nn.Module):
    def __init__(self, z_channels=512):
        super().__init__()
        self.enc = nn.Sequential(
            # 3 → 64
            nn.Conv2d(3, 64, 4, 2, 1),     # 64 x 16 x 16
            nn.ReLU(inplace=True),

            # 64 → 128
            nn.Conv2d(64, 128, 4, 2, 1),   # 128 x 8 x 8
            nn.ReLU(inplace=True),

            # extra depth
            nn.Conv2d(128, 128, 3, 1, 1),  # 128 x 8 x 8
            nn.ReLU(inplace=True),

            # 128 → 256
            nn.Conv2d(128, 256, 4, 2, 1),  # 256 x 4 x 4
            nn.ReLU(inplace=True),

            # more depth at bottleneck
            nn.Conv2d(256, z_channels, 3, 1, 1),  # z_channels x 4 x 4
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.enc(x)  # (B, z_channels, 4, 4)

class Decoder(nn.Module):
    def __init__(self, z_channels=512):
        super().__init__()
        self.dec = nn.Sequential(
            # bottleneck depth
            nn.Conv2d(z_channels, 256, 3, 1, 1),   # 256 x 4 x 4
            nn.ReLU(inplace=True),

            # 4x4 → 8x8
            nn.ConvTranspose2d(256, 256, 4, 2, 1), # 256 x 8 x 8
            nn.ReLU(inplace=True),

            # extra depth
            nn.Conv2d(256, 128, 3, 1, 1),          # 128 x 8 x 8
            nn.ReLU(inplace=True),

            # 8x8 → 16x16
            nn.ConvTranspose2d(128, 128, 4, 2, 1), # 128 x 16 x 16
            nn.ReLU(inplace=True),

            # 16x16 → 32x32
            nn.ConvTranspose2d(128, 64, 4, 2, 1),  # 64 x 32 x 32
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 3, 3, 1, 1),             # 3 x 32 x 32
            nn.Tanh()                               # [-1,1]
        )

    def forward(self, z):
        return self.dec(z)

class AutoEncoder(nn.Module):
    def __init__(self, z_channels=512):
        super().__init__()
        self.encoder = Encoder(z_channels)
        self.decoder = Decoder(z_channels)

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon


In [5]:
def train_autoencoder(autoencoder, train_loader, test_loader,
                      epochs=20, lr=1e-4, weight_decay=1e-5, patience=6):

    autoencoder = autoencoder.to(device)

    opt = torch.optim.Adam(autoencoder.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    best_loss = float("inf")
    wait = 0

    for epoch in range(epochs):
        autoencoder.train()
        train_losses = []

        for x, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            x = x.to(device)
            recon = autoencoder(x)
            loss = loss_fn(recon, x)

            opt.zero_grad()
            loss.backward()
            opt.step()

            train_losses.append(loss.item())

        val_loss = evaluate_recon_loss(autoencoder, test_loader, loss_fn)

        print(f"Epoch {epoch+1}: train={sum(train_losses)/len(train_losses):.4f}, val={val_loss:.4f}")

        if val_loss < best_loss:
            best_loss = val_loss
            wait = 0
            torch.save(autoencoder.state_dict(), "deep_autoencoder.pth")
            print("Saved")
        else:
            wait += 1
            if wait >= patience:
                print("Early stopping")
                break

def evaluate_recon_loss(autoencoder, loader, loss_fn):
    autoencoder.eval()
    losses = []
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            recon = autoencoder(x)
            loss = loss_fn(recon, x)
            losses.append(loss.item())
    return sum(losses) / len(losses)


In [6]:
autoencoder = AutoEncoder()

train_autoencoder(
    autoencoder,
    train_loader, test_loader,
    epochs=1000, lr=1e-4, weight_decay=1e-5,
    patience=20,
)


Epoch 1/1000:   0%|          | 0/782 [00:00<?, ?it/s]

Epoch 1: train=0.4097, val=0.2832
Saved


Epoch 2/1000:   0%|          | 0/782 [00:00<?, ?it/s]

Epoch 2: train=0.2504, val=0.2328
Saved


Epoch 3/1000:   0%|          | 0/782 [00:00<?, ?it/s]

Epoch 3: train=0.2273, val=0.2224
Saved


... (806 outputs left out)

Epoch 407/1000:   0%|          | 0/782 [00:00<?, ?it/s]

Epoch 407: train=0.1164, val=0.1160


Epoch 408/1000:   0%|          | 0/782 [00:00<?, ?it/s]

Epoch 408: train=0.1164, val=0.1159
Early stopping


In [7]:
def load_model(path="full_model.pth"):
    print("\nLoading pretrained ResNet18 model...")
    model = resnet18(weights=ResNet18_Weights.DEFAULT)

    # Modify final layer for 2 classes
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, 10)

    # Freeze all layers except layer3, layer4, fc
    for param in model.parameters():
        param.requires_grad = False
    for layer in [model.layer3, model.layer4, model.fc]:
        for param in layer.parameters():
            param.requires_grad = True

    # Load fine-tuned weights if available
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location=device))
        print(f"Loaded fine-tuned weights from '{path}'")
    else:
        print(f"Warning: '{path}' not found. Model is not fine-tuned yet.")

    return model.to(device)


model = load_model()
model.eval()



Loading pretrained ResNet18 model...
Loaded fine-tuned weights from 'full_model.pth'


/tmp/ipykernel_80192/1370779204.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(path, map_location=device))


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [8]:
os.makedirs("tmp_eval_images", exist_ok=True)

def denormalize(img_tensor):
    img = img_tensor.detach().cpu().numpy()
    img = np.clip(((img * 0.5) + 0.5) * 255, 0, 255).astype(np.uint8)
    img = np.transpose(img, (1, 2, 0))
    return img

def save_tensor_as_jpeg(img_tensor, path, quality=95):
    img = Image.fromarray(denormalize(img_tensor))
    img.save(path, "JPEG", quality=quality)

def save_and_get_size_kb(img_tensor, fname):
    path = os.path.join("tmp_eval_images", fname)
    save_tensor_as_jpeg(img_tensor, path)
    return os.path.getsize(path) / 1024

def evaluate_autoencoder_on_test(autoencoder, model, test_loader):
    criterion_ce = torch.nn.CrossEntropyLoss(reduction="sum")
    autoencoder.eval()
    model.eval()

    total_correct = 0
    total_ce_sum = 0.0
    total_images = 0

    orig_sizes = []
    recon_sizes = []

    idx = 0

    with torch.no_grad():
        for x, y in tqdm(test_loader, desc="Evaluating test set"):
            x = x.to(device)
            y = y.to(device)

            recon = autoencoder(x)
            logits = model(recon)

            total_ce_sum += criterion_ce(logits, y).item()
            preds = torch.argmax(logits, dim=1)
            total_correct += (preds == y).sum().item()
            batch_size = x.size(0)
            total_images += batch_size

            for i in range(batch_size):
                orig_kb = save_and_get_size_kb(x[i], f"orig_{idx}.jpeg")
                recon_kb = save_and_get_size_kb(recon[i], f"recon_{idx}.jpeg")
                orig_sizes.append(orig_kb)
                recon_sizes.append(recon_kb)
                idx += 1

    recon_acc = total_correct / total_images if total_images else 0.0
    recon_CE = total_ce_sum / total_images if total_images else 0.0
    orig_avg_KB = np.mean(orig_sizes) if orig_sizes else 0.0
    recon_avg_KB = np.mean(recon_sizes) if recon_sizes else 0.0
    compression_pct = ((orig_avg_KB - recon_avg_KB) / orig_avg_KB * 100) if orig_avg_KB > 0 else 0.0

    df = pd.DataFrame([{
        "recon_acc": recon_acc,
        "recon_CE": recon_CE,
        "orig_avg_KB": orig_avg_KB,
        "recon_avg_KB": recon_avg_KB,
        "compression_pct": compression_pct,
        "Type": "SimpleAE"
    }])

    df.to_csv("AutoEncoderEval.csv", index=False)
    return df

df_eval = evaluate_autoencoder_on_test(autoencoder, model, test_loader)
print(df_eval)


Evaluating test set:   0%|          | 0/157 [00:00<?, ?it/s]

   recon_acc  recon_CE  orig_avg_KB  recon_avg_KB  compression_pct      Type
0     0.6937  1.151448     1.450461      1.417745         2.255566  SimpleAE
